In [2]:
import numpy as np
from numba import cuda
import time

In [7]:
N = 2048  # Start with 1M for testing

data = np.random.randint(
    0,
    10000,
    N,
    dtype=np.int32
)

In [8]:
from numba import cuda

@cuda.jit
def partition_kernel(arr,
                     pivot,
                     less_flags,
                     greater_flags):

    tid = cuda.threadIdx.x
    idx = cuda.grid(1)

    shared = cuda.shared.array(
        256,
        dtype=np.int32
    )

    if idx < arr.size:
        shared[tid] = arr[idx]

    cuda.syncthreads()

    if idx < arr.size:

        val = shared[tid]

        if val < pivot:
            less_flags[idx] = 1
            greater_flags[idx] = 0

        elif val > pivot:
            less_flags[idx] = 0
            greater_flags[idx] = 1

        else:
            less_flags[idx] = 0
            greater_flags[idx] = 0

In [9]:
def gpu_quicksort(arr):

    if len(arr) <= 1:
        return arr

    pivot = arr[len(arr)//2]

    d_arr = cuda.to_device(arr)

    d_less = cuda.device_array(
        len(arr),
        dtype=np.int32
    )

    d_greater = cuda.device_array(
        len(arr),
        dtype=np.int32
    )

    threads = 256
    blocks = (len(arr)+threads-1)//threads

    partition_kernel[
        blocks,
        threads
    ](
        d_arr,
        pivot,
        d_less,
        d_greater
    )

    less_flags = d_less.copy_to_host()
    greater_flags = d_greater.copy_to_host()

    left = arr[less_flags == 1]
    middle = arr[arr == pivot]
    right = arr[greater_flags == 1]

    return np.concatenate([
        gpu_quicksort(left),
        middle,
        gpu_quicksort(right)
    ])

In [10]:
import time

start = time.time()

result = gpu_quicksort(data)

cuda.synchronize()

elapsed = time.time() - start

print("Execution Time:", elapsed)

print(
    "Correctly Sorted:",
    np.all(result[:-1] <= result[1:])
)

/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 8 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 6 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPe

Execution Time: 0.9950289726257324
Correctly Sorted: True


In [11]:
print("\nFirst 20 Elements Before Sorting:")
print(data[:20])

print("\nFirst 20 Elements After Sorting:")
print(result[:20])

print("\nLast 20 Elements After Sorting:")
print(result[-20:])


First 20 Elements Before Sorting:
[3761 7664  484 7842 5751   91 1256   34 2685  608 2170 6832 4991 2287
 7453 8732  828 1768 1371 1061]

First 20 Elements After Sorting:
[ 1  5 14 15 18 21 25 27 29 32 34 40 43 44 46 50 55 56 62 65]

Last 20 Elements After Sorting:
[9883 9888 9908 9911 9912 9914 9923 9923 9931 9935 9942 9958 9964 9968
 9971 9982 9984 9991 9993 9997]
